# CSA161 — TEST5000 có và không reranking, cấu hình đã khóa
    Chạy trên **notebook của bạn**, **T4x2**, Internet ON, Save Version → Save & Run All.
    Attach3Datasets: **mscoco-2014 có JPEG + dataset_coco.json**, **objectdetectionecache**, **visual-cache gốc**.
    Bật Secret API (mặc định `codex`, sửa cell đầu nếu tên khác).
    Tái dùng best.bin161 đã tải trong phiên lỗi nếu còn, hoặc tải đúng161. Giữ output cũ, ghi folder **_v2**.
    Version143 không trả file cache: tạo lại TESTROI bằng CPU từ visual-cache H5 gốc, dùng đúng mã cũ;
    bắt buộc NPZ/metadata/vocab khớp cả ba SHA đã chốt. Không chạy lại CLIP hay train để tạo cache.
    Nếu đã có cache TEST đúng SHA được mount, điền TEST_ROOT để bỏ qua CPU restore/visual-cache.
    Không cần Dataset checkpoint mới, ROI134, bank hay BLIP. CPU pooling cũ từng khoảng23phút, không đảm bảo thời gian.

    Bộ ứng viên163 là VALID nên không dùng làm TEST. Lượt này sinh5câu cho5000TESTảnh một lần,
    rồi so cùng checkpoint: decoder rank0 và công thức cũ .3decoder+.4CLIP+.3đối tượng.
    Giữ nguyên beam5/max20, CLIP/processor/hệ số như VALID164; không train/tune/thay checkpoint.
    Tham chiếu chỉ dùng chấm điểm sau chọn. Báo cả hai cách dù kết quả tăng hay giảm.
    CIDEr/BLEU4/ROUGE/CHAIR/recall/độ dài; không dùng oracle hay đo loss mới cho reranking.
    TEST đã có các lượt đánh giá trước trong dự án, không gọi là tập kiểm tra chưa từng xem.

    Giới hạn dừng2h, không phải dự báo quota; đo tốc độ từng phase, giữ kết quả dở khi không đủ.
    HF dependencies trong thư mục tạm riêng, không thay Torch. Agent chỉ kiểm dữ liệu/NumPy/static;
    việc tải model, chạy GPU và scorer do người dùng thực hiện. Gửi ZIP review sau chạy.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
COCO_ROOT=Path('/kaggle/input/datasets/vuthetam/mscoco-2014')
DATASET_JSON=COCO_ROOT/'dataset_coco.json'
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
CHECKPOINT_ROOT='' # Optional mounted root containing exact161 best.bin; otherwise download your own exact161.
TEST_ROOT='' # Optional mounted exact TEST cache; otherwise CPU restore from original raw H5, exact byte hashes required.
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
SECRET_NAME='codex' # Enabled API Secret name. Change if yours is KAGGLE_API_TOKEN.
WALL_HOURS=2 # STOP cap, not ETA or billing. Last reported quota16h before VALID164.
OUTPUT=Path('/kaggle/working/csa_ce161_fixed_rerank_test5000_v2')
for p in (COCO_ROOT,DATASET_JSON,DETECTIONS):
    if not p.exists():raise FileNotFoundError('Attach the same2rawDatasets as164 before GPU: '+str(p))
if not TEST_ROOT and not VISUAL_CACHE.is_dir():raise FileNotFoundError('Add THIRD Dataset visual-cache for CPU TEST restoration: '+str(VISUAL_CACHE))
if OUTPUT.exists():raise RuntimeError('Preserve previous output/scores; fresh runtime/path required.')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for immutable inputs/CLIP assets.')
GPU=json.loads(subprocess.check_output([sys.executable,'-c',"import json,torch;print(json.dumps(dict(built=torch.backends.cuda.is_built(),available=torch.cuda.is_available(),count=torch.cuda.device_count())))"],text=True,timeout=60))
if not GPU['built'] or not GPU['available'] or GPU['count']!=2:raise RuntimeError('Select T4x2 and restart runtime.')
OUTPUT.mkdir();SOURCE=Path(tempfile.mkdtemp(prefix='csa-fixed-test-'));LOG=OUTPUT/'notebook.log'
START=time.monotonic();DEADLINE=START+WALL_HOURS*3600
COMMIT='19e7dfe1fb0b9de9f6d047c644a3b429289bb6f4'
HASHES={'experiments/csa_paper/rerank_test_data.py': 'fe357fd8a401d6c9fec9bfb69ffba1ef633c4df2a849c56e71037ee2b2c1038a', 'experiments/csa_paper/run_rerank_test.py': 'eb91f535bffbdc8d670d9852748f1de516fcd7ec80e930ab756dbaa9ff9d6b3a', 'experiments/csa_paper/test_rerank_test_data.py': '51dece1ddd19d5fefca23895d0e04f358a9bf2215b7bf8c062706d98e46de08b', 'experiments/csa_paper/make_rerank_test_notebook.py': 'b1c1d9474072c65feae788c32c47e03ce03e5c497817d79b927e3b3766b5db6b', 'experiments/csa_paper/RERANK_TEST_PROTOCOL.md': '257379f2bc179840c81df3781cfd1d6d0b2365fb65ea1b5c10d65d45c2afa42f', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46', 'experiments/csa_paper/make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/candidate_data.py': 'c9f8cf09d2ab72372eb1fffdc4b56bb097b12e8dd46aac68c60d60b65cf04257', 'experiments/csa_paper/rerank_data.py': '9621a0c3cabc27fcddb7fc891e31bd704c0bf5028721409a9f691d003c397dde', 'experiments/csa_paper/plain_test_data.py': '5416d24ff2df7a34b52e76ab4cd93bbab90f6a0fdc564842c415217ec63514ab', 'experiments/csa_paper/full_xe_decode.py': 'b8e618559feec6361cfb3b4619899e4968ab71ab67e113ff38b6d3cd99cdc235', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/prepare_test_regions.py': '3c61ed502f3091293c537a32ec2c0f82c72f4bf97cc86cfabe4b035c365692b7', 'experiments/csa_paper/prepare_real_assets.py': '44a22d92acd239bdc7a1edbe30984daf0853ac8bc196969dd48b19b78df1cc7b', 'experiments/csa_paper/audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/run_best9_decode.py': 'fdc6b027ac13d510be72aa25a963eb89e5467e09f179bfaf0756469aef77a648', 'experiments/csa_paper/best9_saved_text.json.gz': '42594e90ec2f47646e7e68347a17471213038bc182814ca9d29ea374e8010861', 'experiments/csa_paper/BEST9_SAVED_TEXT_PACKAGE.json': '898d67c3eead66c89f5d2c091b24e5e3eb44d884e86550b2b26f45d6708032e5', 'experiments/csa_paper/SAVE_XE139_PACKAGE.json': '452ee35118a3d0285bc7f522bb8c8fc7a0e3e0dac70cac9c2333e7583cd4e423', 'experiments/csa_paper/full_roi_review_text/input_receipt.json': '18e9481863418c7f2f4c538fa98f3a05377e5b81422cb249d33f81ddefd78f7f', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/multiscore_reranking.py': 'd1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73'}
MAPPING={'experiments/csa_paper/rerank_test_data.py': 'rerank_test_data.py', 'experiments/csa_paper/run_rerank_test.py': 'run_rerank_test.py', 'experiments/csa_paper/test_rerank_test_data.py': 'test_rerank_test_data.py', 'experiments/csa_paper/make_rerank_test_notebook.py': 'make_rerank_test_notebook.py', 'experiments/csa_paper/RERANK_TEST_PROTOCOL.md': 'RERANK_TEST_PROTOCOL.md', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': 'RERANK_TEST_INPUTS.json', 'experiments/csa_paper/make_smoke_notebook.py': 'make_smoke_notebook.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/candidate_data.py': 'candidate_data.py', 'experiments/csa_paper/rerank_data.py': 'rerank_data.py', 'experiments/csa_paper/plain_test_data.py': 'plain_test_data.py', 'experiments/csa_paper/full_xe_decode.py': 'full_xe_decode.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/prepare_test_regions.py': 'prepare_test_regions.py', 'experiments/csa_paper/prepare_real_assets.py': 'prepare_real_assets.py', 'experiments/csa_paper/audit_assets.py': 'audit_assets.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/run_best9_decode.py': 'run_best9_decode.py', 'experiments/csa_paper/best9_saved_text.json.gz': 'best9_saved_text.json.gz', 'experiments/csa_paper/BEST9_SAVED_TEXT_PACKAGE.json': 'BEST9_SAVED_TEXT_PACKAGE.json', 'experiments/csa_paper/SAVE_XE139_PACKAGE.json': 'SAVE_XE139_PACKAGE.json', 'experiments/csa_paper/full_roi_review_text/input_receipt.json': 'full_roi_review_text/input_receipt.json', 'captioning/__init__.py': 'captioning/__init__.py', 'captioning/multiscore_reranking.py': 'captioning/multiscore_reranking.py', 'captioning/object_consistency.py': 'captioning/object_consistency.py'}
for n,h in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/{n}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Pinned source differs: '+n)
    p=SOURCE/MAPPING[n];p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES,gpu=GPU),indent=2),encoding='utf-8')
os.environ.update(OMP_NUM_THREADS='1',TOKENIZERS_PARALLELISM='false',USE_TF='0',USE_FLAX='0',HF_HUB_DISABLE_XET='1')
CONFIG=SOURCE/'config.json';CHAIR=SOURCE/'chair';DEPS=SOURCE/'hf_deps'


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Frozen TEST active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_rerank_test_data.py'],timeout=120)
    sys.path.insert(0,str(SOURCE))
    locked=json.loads((SOURCE/'RERANK_TEST_INPUTS.json').read_text(encoding='utf-8'))
    downloads=[]
    if not CHECKPOINT_ROOT:
        from plain_test_data import exact_file
        prior=Path('/kaggle/working/csa_ce161_fixed_rerank_test5000/inputs161')
        if prior.is_dir():
            try:
                reused=exact_file(prior,'best.bin',locked['checkpoint_sha256']);CHECKPOINT_ROOT=str(reused.parent)
                print('Reusing verified161weights already downloaded:',reused,flush=True)
            except ValueError:pass
    if not CHECKPOINT_ROOT:
        CHECKPOINT_ROOT=str(OUTPUT/'inputs161');Path(CHECKPOINT_ROOT).mkdir()
        downloads.append((locked['checkpoint_version'],CHECKPOINT_ROOT,r'(^|/)best\.bin$'))
    restore_cache=not bool(TEST_ROOT)
    if restore_cache:TEST_ROOT=str(OUTPUT/'restored_test_cache')
    if downloads:
        run([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],timeout=300)
        routing="""
from types import SimpleNamespace
from unittest.mock import patch
from kaggle.api.kaggle_api_extended import KaggleApi
import tempfile
api=KaggleApi()
for version in ('161',):
    with tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:
        endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output
        endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)
        api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/'+version,path=directory)
        request=endpoint.call_args.args[0]
        assert (request.user_name,request.kernel_slug,request.version_label)==('ducanh2403','2-vi-cai-kia-qua-nhieu','v'+version)
print('Exact own161 API routing mock PASS; no143cache download.',flush=True)
"""
        run([sys.executable,'-c',routing],timeout=60)
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret(SECRET_NAME).strip()
        if not token:raise RuntimeError('Enabled API Secret empty.')
        env=os.environ.copy()
        for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):env.pop(key,None)
        env['KAGGLE_API_TOKEN']=token;env['PYTHONUNBUFFERED']='1'
        for version,directory,pattern in downloads:
            if version!='ducanh2403/2-vi-cai-kia-qua-nhieu/161':raise ValueError('No latest/version fallback.')
            print('Downloading frozen own input',version,flush=True)
            command=[sys.executable,'-c','from kaggle.cli import main; main()','kernels','output',version,'-p',directory,'--file-pattern',pattern]
            subprocess.run(command,env=env,check=True,timeout=900)
        env.pop('KAGGLE_API_TOKEN',None);del env,token
    cfg=dict(checkpoint_root=CHECKPOINT_ROOT,test_root=TEST_ROOT,coco_root=str(COCO_ROOT),dataset_json=str(DATASET_JSON),detections=str(DETECTIONS),
        output=str(OUTPUT),chair=str(CHAIR),model_cache=str(SOURCE/'model_cache'),visual_cache=str(VISUAL_CACHE),
        test_cache_mode='CPU_original_H5_exact_byte_match' if restore_cache else 'verified_mounted')
    CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
    if restore_cache:
        remaining=DEADLINE-time.monotonic()-1800
        if remaining<1800:raise TimeoutError('Insufficient CPU restoration/inference reserve; preserve weights.')
        run([sys.executable,'-u',SOURCE/'run_rerank_test.py','rebuild_cache','--config',CONFIG],timeout=min(3720,remaining))
    run([sys.executable,'-u',SOURCE/'run_rerank_test.py','prepare','--config',CONFIG],timeout=min(900,DEADLINE-time.monotonic()))
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;from pycocoevalcap.cider.cider import Cider;from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    assert hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()=='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'
    (OUTPUT/'metric_preflight.json').write_text(json.dumps(dict(passed=True,dependencies_only=True,no_test_metrics_yet=True)),encoding='utf-8')
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'transformers==4.57.1','tokenizers==0.22.1','huggingface-hub==0.36.2','safetensors==0.6.2'],timeout=300)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"from importlib.metadata import version;expected={'transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.2','safetensors':'0.6.2'};assert all(version(k)==v for k,v in expected.items());print('Isolated pinned HF dependencies PASS')"],timeout=60)
    if DEADLINE-time.monotonic()<2400:raise TimeoutError('Insufficient model/decode/scoring reserve; preserve inputs.')
    run([sys.executable,'-u',SOURCE/'run_rerank_test.py','prefetch','--config',CONFIG],timeout=min(1200,DEADLINE-time.monotonic()-1800))
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    cfg['phase_seconds']=DEADLINE-time.monotonic()-1800
    if cfg['phase_seconds']<600:raise TimeoutError('Insufficient TEST decode and scoring reserve.')
    CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
    run([*launch,SOURCE/'run_rerank_test.py','decode','--config',CONFIG],timeout=cfg['phase_seconds']+60)
    run([sys.executable,'-u',SOURCE/'run_rerank_test.py','text_preflight','--config',CONFIG],timeout=180)
    cfg['phase_seconds']=DEADLINE-time.monotonic()-900
    if cfg['phase_seconds']<300:raise TimeoutError('Insufficient CLIP/metric reserve; beams preserved.')
    CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
    run([*launch,SOURCE/'run_rerank_test.py','worker','--config',CONFIG],timeout=cfg['phase_seconds']+60)
    run([sys.executable,'-u',SOURCE/'run_rerank_test.py','score','--config',CONFIG],timeout=min(900,DEADLINE-time.monotonic()))
    complete=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))['status']=='CSA161_FIXED_RERANK_TEST5000_COMPLETE'
    if not complete:raise RuntimeError('Incomplete locked TEST evaluation')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,test_used=True,tuned_on_test=False)),encoding='utf-8')
    archive=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',archive,flush=True)
    print('KEEP WHOLE OUTPUT:',OUTPUT,flush=True)
    print('COMPLETE frozen same-checkpoint TEST5000 comparison' if complete else 'INCOMPLETE; preserve partial beams/scores/logs; no automatic retry.',flush=True)
